# Explore change data

In [21]:
import psycopg2
from dotenv import load_dotenv
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt

dotenv_path = Path().resolve().parent.parent / ".env"
load_dotenv(dotenv_path, override=True)

DB_USER = os.environ.get("DB_USER")
DB_PASS = os.environ.get("DB_PASS")
DB_NAME = os.environ.get("DB_NAME")
DB_HOST = os.environ.get("DB_HOST")
DB_PORT = os.environ.get("DB_PORT")

conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASS,
    host=DB_HOST,
    port=DB_PORT
)

revision_table = "revision_sample_30"
change_table = "value_change_sample_30"

In [22]:
def query_to_df(query, connection):
    try:
        with connection.cursor() as cur:
            cur.execute(query)
            
            if cur.description is not None:
                # Get column names
                colnames = [desc[0] for desc in cur.description]
                # Fetch all rows
                rows = cur.fetchall()
                # Return as Poras DataFrame
                return pd.DataFrame(rows, columns=colnames)
            else:
                print('Query did not return any rows')
                return pd.DataFrame()
    except Exception as e:
        raise e

### Stats 30 files

In [15]:
# number of processed files
df = query_to_df(f"""
    SELECT COUNT(DISTINCT file_path) as count_files
    FROM {revision_table}
    """, conn)
print('Number of processed files:', f"{df.iloc[0]['count_files']:,}")

# number of entities
df = query_to_df(f"""
    SELECT COUNT(DISTINCT entity_id) as count_entities
    FROM {revision_table}""", conn)
print('Number of entities:', f"{df.loc[0,'count_entities']:,}")

# avg. number of changes per revision_sample
df = query_to_df(f"""
    SELECT AVG(count_changes) as avg_count_changes
    FROM(
        SELECT COUNT(*) as count_changes
        FROM {revision_table} r JOIN {change_table} c ON c.revision_id = r.revision_id
        GROUP BY r.entity_id             
    ) AS subquery
""", conn)
print('Avg. number of changes per entity:', f"{df.loc[0,'avg_count_changes']:.2f}")

# number of revisions
df = query_to_df(f"""
    SELECT count(*) as count_revisions
    FROM {revision_table}
""", conn)
print('Number of revisions:', f"{df.loc[0,'count_revisions']:,}")

# number of changes
df = query_to_df(f"""
    SELECT count(*) as count_changes
    FROM {change_table}
""", conn)
print('Number of changes:', f"{df.loc[0,'count_changes']:,}")

# number of CREATE changes
df = query_to_df(f"""
    select count(*) as count_creates
    from {change_table}
    where action = 'CREATE'
""", conn)
print('Number of CREATE changes:', f"{df.loc[0,'count_creates']:,}")

# number of DELETE changes
df = query_to_df(f"""
    select count(*) as count_deletes
    from {change_table}
    where action = 'DELETE'
""", conn)
print('Number of DELETE changes:', f"{df.loc[0,'count_deletes']:,}")

# number of UPDATE changes
df = query_to_df(f"""
    select count(*) as count_updates
    from {change_table}
    where action = 'UPDATE'
""", conn)
print('Number of UPDATE changes:', f"{df.loc[0,'count_updates']:,}")

df = query_to_df(f"""
    select count(*) as count_typo
    from {change_table}
    where typo = TRUE and action = 'UPDATE'
""", conn)
print('Number of typo changes:', f"{df.loc[0,'count_typo']:,}")

df = query_to_df(f"""
    select count(*) as count_formatting
    from {change_table}
    where formatting = TRUE and action = 'UPDATE'
""", conn)
print('Number of formatting changes:', f"{df.loc[0,'count_formatting']:,}")

df = query_to_df(f"""
    select count(*) as count_value_refinement
    from {change_table}
    where value_refinement = TRUE and action = 'UPDATE'
""", conn)
print('Number of value_refinement changes:', f"{df.loc[0,'count_value_refinement']:,}")

df = query_to_df(f"""
    select count(*) as count_value_unrefinement
    from {change_table}
    where value_unrefinement = TRUE and action = 'UPDATE'
""", conn)
print('Number of value_unrefinement changes:', f"{df.loc[0,'count_value_unrefinement']:,}")

df = query_to_df(f"""
    select count(*) as count_reverted_edit
    from {change_table}
    where reverted_edit = TRUE and reversion = FALSE and action = 'UPDATE'
""", conn)
print('Number of reverted_edit changes:', f"{df.loc[0,'count_reverted_edit']:,}")

df = query_to_df(f"""
    select count(*) as count_reversion
    from {change_table}
    where reversion = TRUE and reverted_edit = FALSE and action = 'UPDATE'
""", conn)
print('Number of reversion changes:', f"{df.loc[0,'count_reversion']:,}")

df = query_to_df(f"""
    select count(*) as count
    from {change_table}
    where reversion = TRUE and reverted_edit = TRUE and action = 'UPDATE'
""", conn)
print('Number of reversion and reverted_edit changes:', f"{df.loc[0,'count']:,}")

df = query_to_df(f"""
    SELECT count(*) as unclassified_updates
    FROM {change_table}
    WHERE action = 'UPDATE'
      AND NOT (formatting OR typo OR value_refinement OR 
               value_unrefinement OR reverted_edit OR reversion)
""", conn)
print('Number of UPDATE changes not classified:', f"{df.loc[0,'unclassified_updates']:,}")

Number of processed files: 30
Number of entities: 40,888
Avg. number of changes per entity: 233.26
Number of revisions: 6,372,017
Number of changes: 9,537,716
Number of CREATE changes: 7,255,985
Number of DELETE changes: 1,474,603
Number of UPDATE changes: 807,128
Number of typo changes: 921
Number of formatting changes: 39,206
Number of value_refinement changes: 37,021
Number of value_unrefinement changes: 29,571
Number of reverted_edit changes: 131,667
Number of reversion changes: 113,829
Number of reversion and reverted_edit changes: 10,595
Number of UPDATE changes not classified: 444,318


### Run log - SQL classification

2025-10-14 11:18:17,871 [INFO] Running change classification: 9,537,716 changes, 6,372,017 revisions, 30 files, 40,888 entities.

2025-10-14 11:29:57,143 [INFO] Finished reverted edit classification. Took 699.2621438503265 seconds. (11 min)

2025-10-14 11:30:29,214 [INFO] Finished formatting classification. Took 32.06814408302307 seconds. 

2025-10-14 11:31:15,626 [INFO] Finished typo classification. Took 46.40333914756775 seconds.

2025-10-14 11:32:58,746 [INFO] Finished value refinement classification. Took 103.11799001693726 seconds. (1.7 min)

2025-10-14 11:34:57,608 [INFO] Finished value unrefinement classification. Took 118.85888314247131 seconds.  (1.9 min)

**Total updates: 807128**

**Total classified: 362810 (45% approx)**

**Unclassified updates:** 807128 - 362810 = **444,318 (ok)**


### Stats 50 files

In [ ]:
revision_table = 'revision_sample_50'
change_table = 'value_change_sample_50'

# number of processed files
df = query_to_df(f"""
    SELECT COUNT(DISTINCT file_path) as count_files
    FROM {revision_table}
    """, conn)
print('Number of processed files:', f"{df.iloc[0]['count_files']:,}")

# number of entities
df = query_to_df(f"""
    SELECT COUNT(DISTINCT entity_id) as count_entities
    FROM {revision_table}""", conn)
print('Number of entities:', f"{df.loc[0,'count_entities']:,}")

# avg. number of changes per revision_sample
df = query_to_df(f"""
    SELECT AVG(count_changes) as avg_count_changes
    FROM(
        SELECT COUNT(*) as count_changes
        FROM {revision_table} r JOIN {change_table} c ON c.revision_id = r.revision_id
        GROUP BY r.entity_id             
    ) AS subquery
""", conn)
print('Avg. number of changes per entity:', f"{df.loc[0,'avg_count_changes']:.2f}")

# number of revisions
df = query_to_df(f"""
    SELECT count(*) as count_revisions
    FROM {revision_table}
""", conn)
print('Number of revisions:', f"{df.loc[0,'count_revisions']:,}")

# number of changes
df = query_to_df(f"""
    SELECT count(*) as count_changes
    FROM {change_table}
""", conn)
print('Number of changes:', f"{df.loc[0,'count_changes']:,}")

# number of CREATE changes
df = query_to_df(f"""
    select count(*) as count_creates
    from {change_table}
    where action = 'CREATE'
""", conn)
print('Number of CREATE changes:', f"{df.loc[0,'count_creates']:,}")

# number of DELETE changes
df = query_to_df(f"""
    select count(*) as count_deletes
    from {change_table}
    where action = 'DELETE'
""", conn)
print('Number of DELETE changes:', f"{df.loc[0,'count_deletes']:,}")

# number of UPDATE changes
df = query_to_df(f"""
    select count(*) as count_updates
    from {change_table}
    where action = 'UPDATE'
""", conn)
print('Number of UPDATE changes:', f"{df.loc[0,'count_updates']:,}")

df = query_to_df(f"""
    select count(*) as count_typo
    from {change_table}
    where typo = TRUE and action = 'UPDATE'
""", conn)
print('Number of typo changes:', f"{df.loc[0,'count_typo']:,}")

df = query_to_df(f"""
    select count(*) as count_formatting
    from {change_table}
    where formatting = TRUE and action = 'UPDATE'
""", conn)
print('Number of formatting changes:', f"{df.loc[0,'count_formatting']:,}")

df = query_to_df(f"""
    select count(*) as count_value_refinement
    from {change_table}
    where value_refinement = TRUE and action = 'UPDATE'
""", conn)
print('Number of value_refinement changes:', f"{df.loc[0,'count_value_refinement']:,}")

df = query_to_df(f"""
    select count(*) as count_value_unrefinement
    from {change_table}
    where value_unrefinement = TRUE and action = 'UPDATE'
""", conn)
print('Number of value_unrefinement changes:', f"{df.loc[0,'count_value_unrefinement']:,}")

df = query_to_df(f"""
    select count(*) as count_reverted_edit
    from {change_table}
    where reverted_edit = TRUE and reversion = FALSE and action = 'UPDATE'
""", conn)
print('Number of reverted_edit changes:', f"{df.loc[0,'count_reverted_edit']:,}")

df = query_to_df(f"""
    select count(*) as count_reversion
    from {change_table}
    where reversion = TRUE and reverted_edit = FALSE and action = 'UPDATE'
""", conn)
print('Number of reversion changes:', f"{df.loc[0,'count_reversion']:,}")

df = query_to_df(f"""
    select count(*) as count
    from {change_table}
    where reversion = TRUE and reverted_edit = TRUE and action = 'UPDATE'
""", conn)
print('Number of reversion and reverted_edit changes:', f"{df.loc[0,'count']:,}")

df = query_to_df(f"""
    SELECT count(*) as unclassified_updates
    FROM {change_table}
    WHERE action = 'UPDATE'
      AND NOT (formatting OR typo OR value_refinement OR 
               value_unrefinement OR reverted_edit OR reversion)
""", conn)
print('Number of UPDATE changes not classified:', f"{df.loc[0,'unclassified_updates']:,}")


Number of processed files: 50
Number of entities: 131,075
Avg. number of changes per entity: 178.68
Number of revisions: 16,423,250
Number of changes: 23,421,049
Number of CREATE changes: 18,618,205
Number of DELETE changes: 3,124,165
Number of UPDATE changes: 1,678,679
Number of typo changes: 2,749
Number of formatting changes: 130,592
Number of value_refinement changes: 126,990
Number of value_unrefinement changes: 117,208
Number of reverted_edit changes: 131,667
Number of reversion changes: 113,829
Number of reversion and reverted_edit changes: 10,595
Number of UPDATE changes not classified: 1,045,049
Number of UPDATE changes: 425,653


In [ ]:
conn.close()